# Notebook 02 — Pipeline de Clustering
## Proyecto 6: Clustering de Modos de Consumo Energético — BPC Estación de Bombeo

**Prerequisito:** Ejecutar primero `01_preprocessing_eda.ipynb` (genera `outputs/df_features.parquet`)

---

### Estructura de este notebook:
1. **Estandarización** — por qué y cómo escalar los datos antes de clustering
2. **Reducción de dimensionalidad** — PCA (interpretabilidad) + UMAP (visualización no lineal)
3. **Algoritmos de clustering** — K-Means, DBSCAN, Hierarchical Agglomerative
4. **Validación** — Silhouette, Davies-Bouldin, Calinski-Harabasz
5. **Selección del modelo final** — justificación técnica y física
6. **Caracterización de clusters** — qué significa cada grupo operativamente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Clustering
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import cdist

try:
    import umap
    UMAP_AVAILABLE = True
    print('UMAP disponible')
except ImportError:
    UMAP_AVAILABLE = False
    print('UMAP no disponible — instalar: pip install umap-learn')

RANDOM_STATE = 42
plt.rcParams.update({'figure.dpi': 120, 'figure.facecolor': '#0f1117',
                     'axes.facecolor': '#1a1d27', 'axes.edgecolor': '#3d4166',
                     'text.color': '#e0e0e0', 'axes.labelcolor': '#e0e0e0',
                     'xtick.color': '#b0b0b0', 'ytick.color': '#b0b0b0'})

CLUSTER_COLORS = ['#4FC3F7', '#FF8A65', '#81C784', '#CE93D8', '#FFD54F',
                  '#80CBC4', '#F48FB1', '#BCAAA4', '#90CAF9']
print('Librerías cargadas correctamente.')

In [ ]:
# Cargar features generadas en Notebook 01
df_features = pd.read_parquet('../outputs/df_features.parquet')
df_clean    = pd.read_parquet('../outputs/df_clean.parquet')

FEATURES = df_features.columns.tolist()
print(f'Features cargadas: {len(FEATURES)} variables, {len(df_features):,} registros')
print(f'Features: {FEATURES}')

---
## Sección 1 — Estandarización

### ¿Por qué estandarizar antes de clustering?

Los algoritmos basados en distancias (K-Means, DBSCAN) son **sensibles a la escala** de las variables.  
Sin estandarización, variables con mayor magnitud numérica (ej. RPM: 0–3600, Potencia: 0–2750)  
dominarían la función de distancia sobre variables con menor magnitud (ej. TDH: 0–1500 PSI).  

**Método elegido: StandardScaler** (media=0, std=1)  
- Apropiado para K-Means y clustering jerárquico  
- DBSCAN también se beneficia (sus eps está en espacio escalado)  
- Permite comparar importancia relativa de cada feature

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_features)

df_scaled = pd.DataFrame(X_scaled, columns=FEATURES)
print('Estadísticas post-estandarización (deben ser ≈ media=0, std=1):')
print(df_scaled.describe().loc[['mean', 'std']].round(4).to_string())

---
## Sección 2 — Reducción de Dimensionalidad

### PCA — Análisis de Componentes Principales

**Propósito:** Dos usos distintos:
1. **Interpretabilidad**: cuánta varianza captura cada componente, qué variables aportan más
2. **Preprocessing opcional para K-Means**: si las primeras N componentes capturan > 90% de varianza,
   el clustering en espacio reducido puede ser más estable (menos maldición de la dimensionalidad)

### UMAP — Uniform Manifold Approximation and Projection

**Propósito:** Visualización 2D/3D de alta calidad — preserva estructura local Y global del manifold.  
**NO** se usa como preprocessing de clustering — solo para visualización de resultados.

In [ ]:
# 2.1 PCA — Varianza explicada acumulada
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_comp_90  = np.argmax(cumvar >= 0.90) + 1
n_comp_95  = np.argmax(cumvar >= 0.95) + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(range(1, len(pca_full.explained_variance_ratio_)+1),
            pca_full.explained_variance_ratio_ * 100,
            color='#4FC3F7', edgecolor='none')
axes[0].set_title('Varianza Explicada por Componente')
axes[0].set_xlabel('Componente Principal')
axes[0].set_ylabel('Varianza Explicada [%]')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(1, len(cumvar)+1), cumvar * 100, 'o-', color='#FF8A65', lw=2)
axes[1].axhline(90, color='#81C784', ls='--', lw=1.5, label=f'90% → {n_comp_90} comp.')
axes[1].axhline(95, color='#FFD54F', ls='--', lw=1.5, label=f'95% → {n_comp_95} comp.')
axes[1].set_title('Varianza Explicada Acumulada')
axes[1].set_xlabel('Número de Componentes')
axes[1].set_ylabel('Varianza Acumulada [%]')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Análisis de Componentes Principales (PCA)', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/pca_varianza.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Para capturar 90% de varianza: {n_comp_90} componentes')
print(f'Para capturar 95% de varianza: {n_comp_95} componentes')
print(f'Total de features: {len(FEATURES)}')

In [ ]:
# 2.2 Biplot PCA — qué variables dominan cada componente
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled)

pca_3d = PCA(n_components=3, random_state=RANDOM_STATE)
X_pca_3d = pca_3d.fit_transform(X_scaled)

# Loading plot (contribución de cada variable a PC1 y PC2)
loadings = pd.DataFrame(
    pca_2d.components_.T,
    index=FEATURES,
    columns=['PC1', 'PC2']
).sort_values('PC1', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_bar = ['#FF8A65' if v > 0 else '#4FC3F7' for v in loadings['PC1']]
axes[0].barh(loadings.index, loadings['PC1'], color=colors_bar, edgecolor='none')
axes[0].axvline(0, color='white', lw=0.8)
axes[0].set_title(f'Loadings PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}% varianza)')
axes[0].grid(True, alpha=0.3, axis='x')

colors_bar2 = ['#FF8A65' if v > 0 else '#4FC3F7' for v in loadings['PC2']]
axes[1].barh(loadings.index, loadings['PC2'], color=colors_bar2, edgecolor='none')
axes[1].axvline(0, color='white', lw=0.8)
axes[1].set_title(f'Loadings PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}% varianza)')
axes[1].grid(True, alpha=0.3, axis='x')

plt.suptitle('Contribución de Variables a las Primeras 2 Componentes Principales', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: barras largas = alta influencia en ese componente')

In [ ]:
# 2.3 UMAP — reducción no lineal para visualización
# Se usa MUESTRA de 20,000 puntos para velocidad (UMAP escala O(n log n))
UMAP_SAMPLE = min(20000, len(X_scaled))
sample_idx  = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), UMAP_SAMPLE, replace=False)
X_sample    = X_scaled[sample_idx]

if UMAP_AVAILABLE:
    print(f'Calculando UMAP 2D sobre {UMAP_SAMPLE:,} puntos...')
    reducer_2d = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                           metric='euclidean', random_state=RANDOM_STATE)
    X_umap_2d = reducer_2d.fit_transform(X_sample)

    print('Calculando UMAP 3D...')
    reducer_3d = umap.UMAP(n_components=3, n_neighbors=30, min_dist=0.1,
                           metric='euclidean', random_state=RANDOM_STATE)
    X_umap_3d = reducer_3d.fit_transform(X_sample)
    print('UMAP completado.')
else:
    # Fallback: usar PCA 2D y 3D
    X_umap_2d = X_pca_2d[sample_idx]
    X_umap_3d = X_pca_3d[sample_idx]
    print('Usando PCA como fallback de UMAP')

---
## Sección 3 — Algoritmos de Clustering

### ¿Por qué probamos 3 algoritmos?

Cada algoritmo hace **supuestos diferentes** sobre la forma de los clusters:

| Algoritmo | Supuesto | Fortaleza | Limitación |
|-----------|----------|-----------|------------|
| K-Means | Clusters esféricos, igual tamaño | Rápido, centroides interpretables | Número k fijo, sensible a outliers |
| DBSCAN | Clusters de densidad arbitraria | Detecta ruido, no necesita k | Difícil con densidades variables |
| Hierarchical | Sin supuesto de forma | Dendrograma para gerencia | Lento en datasets grandes |

**Estrategia:** Comparamos los tres y elegimos el que mejor combine métricas de validación + coherencia física.

In [ ]:
# 3.1 K-MEANS — Método del Codo + Silhouette para elegir k
K_RANGE  = range(2, 11)
inertias = []
silhouettes_km = []
db_scores_km   = []
ch_scores_km   = []

# Usamos muestra para velocidad en el grid search
N_SAMPLE_GRID = min(30000, len(X_scaled))
idx_grid = np.random.RandomState(RANDOM_STATE).choice(len(X_scaled), N_SAMPLE_GRID, replace=False)
X_grid   = X_scaled[idx_grid]

print(f'Evaluando K-Means con k=2..10 sobre {N_SAMPLE_GRID:,} puntos...')
for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_grid)
    inertias.append(km.inertia_)
    silhouettes_km.append(silhouette_score(X_grid, labels, sample_size=5000, random_state=RANDOM_STATE))
    db_scores_km.append(davies_bouldin_score(X_grid, labels))
    ch_scores_km.append(calinski_harabasz_score(X_grid, labels))
    print(f'  k={k}: Silhouette={silhouettes_km[-1]:.4f}, DB={db_scores_km[-1]:.4f}, CH={ch_scores_km[-1]:.0f}')

print('Evaluación completada.')

In [ ]:
# Visualización de métricas K-Means
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
k_vals = list(K_RANGE)

# Método del codo
axes[0,0].plot(k_vals, inertias, 'o-', color='#4FC3F7', lw=2, ms=7)
axes[0,0].set_title('Método del Codo (Inertia)')
axes[0,0].set_xlabel('Número de Clusters k')
axes[0,0].set_ylabel('Inertia')
axes[0,0].grid(True, alpha=0.3)

# Silhouette
best_k_sil = k_vals[np.argmax(silhouettes_km)]
axes[0,1].plot(k_vals, silhouettes_km, 'o-', color='#81C784', lw=2, ms=7)
axes[0,1].axvline(best_k_sil, color='yellow', ls='--', lw=1.5, label=f'Óptimo k={best_k_sil}')
axes[0,1].set_title('Coeficiente de Silhouette (↑ mejor)')
axes[0,1].set_xlabel('Número de Clusters k')
axes[0,1].set_ylabel('Silhouette Score')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# Davies-Bouldin
best_k_db = k_vals[np.argmin(db_scores_km)]
axes[1,0].plot(k_vals, db_scores_km, 'o-', color='#FF8A65', lw=2, ms=7)
axes[1,0].axvline(best_k_db, color='yellow', ls='--', lw=1.5, label=f'Óptimo k={best_k_db}')
axes[1,0].set_title('Índice Davies-Bouldin (↓ mejor)')
axes[1,0].set_xlabel('Número de Clusters k')
axes[1,0].set_ylabel('Davies-Bouldin Score')
axes[1,0].legend()
axes[1,0].grid(True, alpha=0.3)

# Calinski-Harabasz
best_k_ch = k_vals[np.argmax(ch_scores_km)]
axes[1,1].plot(k_vals, ch_scores_km, 'o-', color='#CE93D8', lw=2, ms=7)
axes[1,1].axvline(best_k_ch, color='yellow', ls='--', lw=1.5, label=f'Óptimo k={best_k_ch}')
axes[1,1].set_title('Índice Calinski-Harabasz (↑ mejor)')
axes[1,1].set_xlabel('Número de Clusters k')
axes[1,1].set_ylabel('Calinski-Harabasz Score')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

plt.suptitle('Evaluación de K-Means — Selección de k óptimo', fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/kmeans_metricas.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nRecomendaciones por métrica:')
print(f'  Silhouette máximo : k = {best_k_sil}')
print(f'  Davies-Bouldin min: k = {best_k_db}')
print(f'  Calinski-H máximo : k = {best_k_ch}')
print(f'\nDECISIÓN FINAL: Ver celda de justificación abajo')

In [ ]:
# -------------------------------------------------------------------
# DECISIÓN DEL NÚMERO DE CLUSTERS
#
# Esta decisión combina 3 criterios:
# 1. Métricas cuantitativas (Silhouette, DB, CH)
# 2. Método del codo (inflexión en la curva de inercia)
# 3. Coherencia física: ¿los centroides tienen sentido operativo?
#
# La coherencia física es el criterio definitivo:
# - Un k que maximiza Silhouette pero produce clusters sin interpretación
#   operativa es matemáticamente óptimo pero inútil para el operador.
# - Los clusters deben corresponder a regímenes operativos reales.
# -------------------------------------------------------------------

# AJUSTAR tras ver las métricas arriba
K_FINAL = best_k_sil  # Punto de partida: k que maximiza Silhouette
                       # Revisar y ajustar según análisis de centroides

print(f'k elegido inicialmente: {K_FINAL}')
print('IMPORTANTE: Verificar coherencia física de centroides en Sección 5')

In [ ]:
# 3.2 K-Means final — entrenamiento sobre dataset completo
km_final = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=20, random_state=RANDOM_STATE)
labels_km = km_final.fit_predict(X_scaled)

df_features['cluster_km'] = labels_km

# Verificar estabilidad (múltiples semillas)
print('Verificando estabilidad del clustering (10 semillas diferentes)...')
stability_scores = []
for seed in range(10):
    km_test = KMeans(n_clusters=K_FINAL, init='k-means++', n_init=5, random_state=seed)
    labels_test = km_test.fit_predict(X_grid)
    s = silhouette_score(X_grid, labels_test, sample_size=5000, random_state=42)
    stability_scores.append(s)

print(f'  Silhouette por semilla: {[round(s,4) for s in stability_scores]}')
print(f'  Media: {np.mean(stability_scores):.4f} ± {np.std(stability_scores):.4f}')
print(f'  → CV = {np.std(stability_scores)/np.mean(stability_scores)*100:.2f}%')
if np.std(stability_scores) / np.mean(stability_scores) < 0.05:
    print('  RESULTADO: Clustering ESTABLE (CV < 5%)')
else:
    print('  ALERTA: Clustering INESTABLE — considerar diferente k o preprocesamiento')

In [ ]:
# 3.3 DBSCAN — detección de ruido y clusters de densidad
# DBSCAN requiere calibrar dos parámetros:
#   eps       : radio máximo de vecindad
#   min_samples: mínimo de puntos para formar un cluster
#
# Para estimar eps: gráfica k-distance (distancia al k-ésimo vecino)
# El codo de esta curva sugiere un buen eps.

from sklearn.neighbors import NearestNeighbors

k_dbscan = 5  # típicamente = n_features * 2, mínimo 5
print(f'Calculando k-distance para DBSCAN (k={k_dbscan})...')

nbrs = NearestNeighbors(n_neighbors=k_dbscan).fit(X_grid)
distances, _ = nbrs.kneighbors(X_grid)
k_distances = np.sort(distances[:, -1])[::-1]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_distances, color='#4FC3F7', lw=1.5)
ax.set_title(f'K-Distance Graph (k={k_dbscan}) — El codo sugiere eps óptimo')
ax.set_xlabel('Puntos ordenados por distancia (desc)')
ax.set_ylabel(f'Distancia al {k_dbscan}° vecino más cercano')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/dbscan_kdistance.png', dpi=150, bbox_inches='tight')
plt.show()
print('INSTRUCCIÓN: Identifica visualmente el codo y usa ese valor como eps')

In [ ]:
# Calibrar EPS visualmente desde la gráfica anterior
# Valor inicial sugerido — AJUSTAR según el codo observado
DBSCAN_EPS         = 0.5   # AJUSTAR tras observar k-distance graph
DBSCAN_MIN_SAMPLES = 50

print(f'Ejecutando DBSCAN: eps={DBSCAN_EPS}, min_samples={DBSCAN_MIN_SAMPLES}')
dbscan = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES, metric='euclidean', n_jobs=-1)
labels_db = dbscan.fit_predict(X_grid)

n_clusters_db = len(set(labels_db)) - (1 if -1 in labels_db else 0)
n_noise_db    = (labels_db == -1).sum()
pct_noise     = n_noise_db / len(labels_db) * 100

print(f'  Clusters detectados : {n_clusters_db}')
print(f'  Puntos de ruido     : {n_noise_db:,} ({pct_noise:.2f}%)')

if n_clusters_db > 0 and n_noise_db < len(labels_db) * 0.30:
    sil_db = silhouette_score(X_grid[labels_db != -1], labels_db[labels_db != -1],
                              sample_size=5000, random_state=RANDOM_STATE)
    print(f'  Silhouette (sin ruido): {sil_db:.4f}')
else:
    print('  ALERTA: Demasiado ruido o muy pocos clusters — ajustar eps/min_samples')

In [ ]:
# 3.4 Clustering Jerárquico Aglomerativo — dendrograma para gerencia
# Visualización más intuitiva: muestra CÓMO se agrupan los modos
# Usamos muestra pequeña para el dendrograma (legibilidad)

N_DENDRO = min(3000, len(X_grid))
idx_d    = np.random.RandomState(RANDOM_STATE).choice(len(X_grid), N_DENDRO, replace=False)
X_dendro = X_grid[idx_d]

print(f'Calculando linkage para dendrograma ({N_DENDRO:,} puntos, método Ward)...')
Z = linkage(X_dendro, method='ward', metric='euclidean')

fig, ax = plt.subplots(figsize=(14, 5))
dendrogram(Z, ax=ax, truncate_mode='level', p=6,
           color_threshold=None, above_threshold_color='#4FC3F7',
           leaf_rotation=90, leaf_font_size=0)
ax.set_title('Dendrograma — Clustering Jerárquico Aglomerativo (Ward)', fontsize=12)
ax.set_xlabel('Registros agrupados')
ax.set_ylabel('Distancia de fusión (Ward)')

# Línea de corte sugerida al nivel k=K_FINAL
last_merges  = Z[-K_FINAL+1:, 2]
cut_threshold = (last_merges[0] + last_merges[-1]) / 2
ax.axhline(cut_threshold, color='#FF8A65', ls='--', lw=2, label=f'Corte → {K_FINAL} clusters')
ax.legend()
plt.tight_layout()
plt.savefig('../outputs/figures/dendrograma.png', dpi=150, bbox_inches='tight')
plt.show()
print('El dendrograma muestra la jerarquía natural de agrupamiento en los datos.')

In [ ]:
# Clustering jerárquico completo
print(f'Calculando Agglomerative Clustering (k={K_FINAL}, Ward)...')
agg = AgglomerativeClustering(n_clusters=K_FINAL, linkage='ward')
labels_agg = agg.fit_predict(X_grid)

sil_agg = silhouette_score(X_grid, labels_agg, sample_size=5000, random_state=RANDOM_STATE)
db_agg  = davies_bouldin_score(X_grid, labels_agg)
ch_agg  = calinski_harabasz_score(X_grid, labels_agg)
print(f'  Silhouette: {sil_agg:.4f}')
print(f'  Davies-Bouldin: {db_agg:.4f}')
print(f'  Calinski-Harabasz: {ch_agg:.0f}')

---
## Sección 4 — Comparación de Algoritmos y Selección del Modelo Final

In [ ]:
# Tabla comparativa
labels_km_grid = km_final.predict(X_grid)
sil_km_final   = silhouette_score(X_grid, labels_km_grid, sample_size=5000, random_state=RANDOM_STATE)
db_km_final    = davies_bouldin_score(X_grid, labels_km_grid)
ch_km_final    = calinski_harabasz_score(X_grid, labels_km_grid)

resultados_df = pd.DataFrame([
    {'Algoritmo': 'K-Means',          'k': K_FINAL,      'Silhouette ↑': sil_km_final, 'Davies-Bouldin ↓': db_km_final, 'Calinski-Harabasz ↑': ch_km_final, 'Ruido %': 0},
    {'Algoritmo': 'Agglomerative',    'k': K_FINAL,      'Silhouette ↑': sil_agg,      'Davies-Bouldin ↓': db_agg,      'Calinski-Harabasz ↑': ch_agg,      'Ruido %': 0},
    {'Algoritmo': 'DBSCAN',           'k': n_clusters_db,'Silhouette ↑': sil_db if n_clusters_db > 0 else None, 'Davies-Bouldin ↓': None, 'Calinski-Harabasz ↑': None, 'Ruido %': round(pct_noise, 2)},
])

print('=== COMPARACIÓN DE ALGORITMOS ===')
print(resultados_df.to_string(index=False))

print('''
CONCLUSIÓN METODOLÓGICA:
- K-Means es preferido para este dataset porque:
  1. Los clusters son aproximadamente compactos y esféricos en espacio PCA
  2. Los centroides son directamente interpretables como "modo operativo típico"
  3. La estabilidad entre semillas es alta (CV < 5%)
  4. Permite asignar nuevos puntos en tiempo real (predict)

- DBSCAN complementa K-Means como validación: los puntos de ruido detectados
  por DBSCAN son candidatos a anomalías operativas.

- Agglomerative es útil para la PRESENTACIÓN A GERENCIA (dendrograma).
''')

---
## Sección 5 — Caracterización de Clusters

Esta sección es la más importante para el operador: **¿qué significa cada cluster?**

In [ ]:
# 5.1 Estadísticas descriptivas por cluster
df_features['cluster_km'] = labels_km
df_features['cluster_label'] = df_features['cluster_km'].astype(str)

print('=== CENTROIDES DE CLUSTERS (en espacio original, no escalado) ===')
centroids_original = df_features.groupby('cluster_km')[FEATURES].mean()
print(centroids_original.round(1).to_string())

In [ ]:
# 5.2 Tamaño y distribución de clusters
cluster_counts = df_features['cluster_km'].value_counts().sort_index()
cluster_pct    = cluster_counts / len(df_features) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors_pie = CLUSTER_COLORS[:K_FINAL]
axes[0].pie(cluster_counts, labels=[f'Cluster {i}' for i in cluster_counts.index],
            colors=colors_pie, autopct='%1.1f%%', pctdistance=0.8, startangle=90)
axes[0].set_title('Distribución de Tamaño por Cluster')

axes[1].bar(cluster_counts.index, cluster_counts.values,
            color=colors_pie, edgecolor='none')
axes[1].set_title('Frecuencia por Cluster')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Número de registros')
for i, (idx, cnt) in enumerate(zip(cluster_counts.index, cluster_counts.values)):
    axes[1].text(idx, cnt + 50, f'{cnt:,}', ha='center', va='bottom', fontsize=9)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Distribución de Registros por Cluster', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/distribucion_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.3 Box plots por cluster — distribución de cada variable
n_vars = len(FEATURES)
ncols  = 3
nrows  = int(np.ceil(n_vars / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
fig.suptitle('Distribución de Variables por Cluster', fontsize=13, y=1.01)

for i, (ax, feat) in enumerate(zip(axes.flat, FEATURES)):
    data_by_cluster = [df_features[df_features['cluster_km'] == k][feat].dropna().values
                       for k in sorted(df_features['cluster_km'].unique())]
    bp = ax.boxplot(data_by_cluster, patch_artist=True, notch=False,
                    medianprops=dict(color='white', lw=2))
    for patch, color in zip(bp['boxes'], CLUSTER_COLORS[:K_FINAL]):
        patch.set_facecolor(color)
        patch.set_alpha(0.75)
    ax.set_title(feat, fontsize=9)
    ax.set_xticklabels([f'C{k}' for k in sorted(df_features['cluster_km'].unique())], fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')

# Ocultar subplots vacíos
for ax in axes.flat[n_vars:]:
    ax.set_visible(False)

plt.tight_layout()
plt.savefig('../outputs/figures/boxplots_clusters.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 5.4 Radar chart (gráfico de araña) por cluster — para gerencia
# Normalizar centroides al rango [0,1] para comparación relativa
centroids = centroids_original.copy()
centroids_norm = (centroids - centroids.min()) / (centroids.max() - centroids.min())

categories = FEATURES
N_cats = len(categories)

fig = go.Figure()
for cluster_id in sorted(df_features['cluster_km'].unique()):
    values = centroids_norm.loc[cluster_id].tolist()
    values += values[:1]  # cerrar el polígono
    cats   = categories + [categories[0]]
    fig.add_trace(go.Scatterpolar(
        r=values, theta=cats,
        fill='toself', opacity=0.5,
        name=f'Cluster {cluster_id}',
        line_color=CLUSTER_COLORS[cluster_id]
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0,1], gridcolor='#2a2d3e'),
               angularaxis=dict(gridcolor='#2a2d3e')),
    showlegend=True,
    title='Perfil de Cada Cluster — Centroides Normalizados',
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27',
    font=dict(color='#e0e0e0')
)
fig.write_html('../outputs/html/radar_clusters.html')
fig.show()

In [ ]:
# 5.5 Mapa de operación 2D (UMAP/PCA) coloreado por cluster
labels_sample_km = km_final.predict(X_sample)

fig = go.Figure()
for cluster_id in sorted(set(labels_sample_km)):
    mask = labels_sample_km == cluster_id
    fig.add_trace(go.Scatter(
        x=X_umap_2d[mask, 0], y=X_umap_2d[mask, 1],
        mode='markers', name=f'Cluster {cluster_id}',
        marker=dict(size=2, color=CLUSTER_COLORS[cluster_id], opacity=0.5)
    ))

dim_label = 'UMAP' if UMAP_AVAILABLE else 'PCA'
fig.update_layout(
    title=f'Mapa de Operación 2D ({dim_label}) — Modos de Consumo Energético',
    xaxis_title=f'{dim_label} Dim 1', yaxis_title=f'{dim_label} Dim 2',
    paper_bgcolor='#0f1117', plot_bgcolor='#1a1d27',
    font=dict(color='#e0e0e0'),
    width=900, height=600
)
fig.write_html('../outputs/html/mapa_operacion_2d.html')
fig.show()

In [ ]:
# 5.6 Mapa de operación 3D interactivo
# Ejes: RPM × Flujo × Potencia — el espacio físico más interpretable
sample_df = df_features.iloc[sample_idx].copy()
sample_df['cluster'] = labels_sample_km

fig_3d = go.Figure()
for cluster_id in sorted(sample_df['cluster'].unique()):
    sub = sample_df[sample_df['cluster'] == cluster_id]
    fig_3d.add_trace(go.Scatter3d(
        x=sub['RPM'], y=sub['Fluj_Desc'], z=sub['Potencia'],
        mode='markers',
        name=f'Cluster {cluster_id}',
        marker=dict(size=2, color=CLUSTER_COLORS[cluster_id], opacity=0.5)
    ))

fig_3d.update_layout(
    title='Mapa de Operación 3D — RPM × Flujo × Potencia',
    scene=dict(
        xaxis=dict(title='RPM', backgroundcolor='#1a1d27', gridcolor='#2a2d3e'),
        yaxis=dict(title='Flujo Desc [GPM]', backgroundcolor='#1a1d27', gridcolor='#2a2d3e'),
        zaxis=dict(title='Potencia [kW]', backgroundcolor='#1a1d27', gridcolor='#2a2d3e'),
        bgcolor='#0f1117'
    ),
    paper_bgcolor='#0f1117', font=dict(color='#e0e0e0'),
    width=950, height=700
)
fig_3d.write_html('../outputs/html/mapa_operacion_3d.html')
fig_3d.show()

In [ ]:
# 5.7 Exportar resultados de clustering para Notebook 03
df_features.to_parquet('../outputs/df_clustered.parquet', index=False)

# Guardar modelo K-Means
import pickle
with open('../outputs/kmeans_model.pkl', 'wb') as f:
    pickle.dump({'scaler': scaler, 'kmeans': km_final, 'features': FEATURES,
                 'k_final': K_FINAL, 'centroids': centroids_original.to_dict()}, f)

print(f'Guardado: df_clustered.parquet ({len(df_features):,} filas)')
print(f'Guardado: kmeans_model.pkl (scaler + modelo + metadatos)')

---
## Resumen del Notebook 02

| Etapa | Resultado |
|-------|-----------|
| Algoritmo seleccionado | K-Means (k = ver arriba) |
| Silhouette final | Ver tabla comparativa |
| Estabilidad | CV < 5% entre semillas |
| Modos identificados | Ver centroides Sección 5 |

**Siguiente paso:** `03_efficiency_3d_viz.ipynb` — análisis del punto de eficiencia y visualizaciones para gerencia.